# IndianLegal-LLM — QLoRA fine-tune (cloud T4)

Fine-tunes an **Apache-2.0** base into an Indian-law assistant whose outputs pass the
Hour-4 **citation guard** (verbatim quote + `[chunk_id]` + paragraph pinpoint). Only a
small **LoRA adapter** (50–200 MB, MIT) is exported — we never redistribute base weights.

**Base (verified Apache-2.0 on the HF model card):**
- Train: `unsloth/Qwen3-4B-Instruct-2507-bnb-4bit` (bnb-4bit, for Unsloth QLoRA on a free T4)
- Serve: `Qwen/Qwen3-4B-Instruct-2507` + the LoRA adapter
- Do **NOT** use Qwen3.5 (Unsloth flags 4-bit QLoRA as not recommended there).
- Phi-4 (MIT) stays the zero-shot serving default; this notebook produces the fine-tuned variant.

## CLAUDE.md compliance
- **§2 (licensing):** base is Apache-2.0; the adapter we train + ship is MIT. Confirm the exact
  repo id on the current model card before use.
- **§5 (cost/bandwidth):** runs **IN THE CLOUD** on a free **T4** (Kaggle/Colab). Data is read
  from the cloud; base weights download **once** on the GPU host. **Never run this on a laptop**
  and never push the corpus/weights/adapter through notebook I/O to a laptop — only the
  50–200 MB adapter comes down.
- **§6 (green build):** the offline skeleton and eval harness are unaffected (stub-pinned).

## Free-tier session limits + the bandwidth rule
- Kaggle: ~30 GPU-hours/week, sessions up to ~12h, T4 16 GB — enough for a 4B QLoRA run.
- Colab free: a single T4, sessions disconnect after idle / ~12h; checkpoint the adapter often.
- QLoRA 4-bit + gradient checkpointing keeps a 4B model within 16 GB.
- **Bandwidth rule:** process the corpus where it lives (cloud). Do not download the corpus or
  base weights to a laptop; only the adapter (50–200 MB) is meant to come down.
- `data/`, `adapters/`, `outputs/`, and `*.safetensors` are gitignored — nothing here is committed.

In [ ]:
# Install (cloud GPU). Unsloth pulls compatible transformers/peft/trl/bitsandbytes.
%pip install -q "unsloth[colab-new]" "trl" "datasets"
# Install this repo so we can reuse the (tested) dataset builder + serving path.
%pip install -q -e ..

In [ ]:
# --- Run configuration ---
TRAIN_MODEL = "unsloth/Qwen3-4B-Instruct-2507-bnb-4bit"  # Apache-2.0, bnb-4bit (QLoRA)
SERVE_BASE  = "Qwen/Qwen3-4B-Instruct-2507"             # Apache-2.0 serving base
ADAPTER_DIR = "adapters/indianlegal-qwen3-4b-lora"       # gitignored; MIT adapter output
MAX_SEQ_LEN = 2048

# SMOKE=True: tiny toy train on the offline stub corpus (acceptance smoke test).
# SMOKE=False: real run — streams a license-clean Supreme Court sample from AWS Open Data.
SMOKE = True

### Step 1 — Build the instruction dataset from the processed corpus
Reuses `indianlegal_llm.finetune.build_instruction_examples`: each example is
(question → grounded answer with a **verbatim quote + `[chunk_id]` + pinpoint**), generated
TEMPLATE-based directly from the license-clean corpus (no external teacher model). Every
example is grounded by construction and passes the Hour-4 guard (asserted in `tests/test_finetune.py`).

In [ ]:
import os
from indianlegal_llm.processing.stub import StubProcessor
from indianlegal_llm.finetune import build_instruction_examples, write_jsonl

processor = StubProcessor()
if SMOKE:
    from indianlegal_llm.ingestion.stub import StubIngestor
    ingestor = StubIngestor()                       # offline, deterministic toy corpus
else:
    from indianlegal_llm.ingestion import get_ingestor
    ingestor = get_ingestor("aws-sc", limit=500)    # real SC judgments, streamed in-cloud

chunks = [c for doc in ingestor.fetch() for c in processor.process(doc)]
examples = build_instruction_examples(chunks)
os.makedirs("data", exist_ok=True)
n = write_jsonl(examples, "data/finetune_train.jsonl")   # gitignored; never committed
print(f"built {n} instruction examples from {len(chunks)} chunks")
print(examples[0].answer if examples else "(no examples)")

### Step 2 — Load the 4-bit base and attach LoRA (QLoRA)
Unsloth loads the bnb-4bit Apache-2.0 base with gradient checkpointing; PEFT adds the LoRA layers.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=TRAIN_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,            # QLoRA 4-bit
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",   # fits a 4B model in 16 GB
    random_state=3407,
)

### Step 3 — Format with the Instruct chat template and train
`max_steps` is tiny for the SMOKE run (acceptance) and larger for a real run.

In [ ]:
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

raw = load_dataset("json", data_files="data/finetune_train.jsonl", split="train")

def to_text(example):
    # Apply the model's own Instruct chat template to the system/user/assistant turns.
    return {"text": tokenizer.apply_chat_template(example["messages"], tokenize=False)}

dataset = raw.map(to_text, remove_columns=raw.column_names)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=10 if SMOKE else 200,
        learning_rate=2e-4,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)
trainer.train()

### Step 4 — Export ONLY the LoRA adapter (50–200 MB)
We save the adapter + tokenizer, NOT the merged base weights (which we never redistribute).

In [ ]:
import subprocess

model.save_pretrained(ADAPTER_DIR)       # adapter_model.safetensors + adapter_config.json only
tokenizer.save_pretrained(ADAPTER_DIR)
print(subprocess.run(["du", "-sh", ADAPTER_DIR], capture_output=True, text=True).stdout)
print("Adapter (MIT) ready. This is the ONLY artifact meant to leave the cloud host.")

### Step 5 — Serve the fine-tuned variant (base + adapter)
On a GPU host, point serving at the Apache-2.0 base + the exported adapter. The Hour-5
`TransformersLLM` loads `SERVE_BASE` 4-bit and layers the LoRA adapter behind the GPU gate;
the Hour-4 guard still applies, so answers are grounded/cited/pinpointed or refused.

In [ ]:
# Equivalent CLI:
#   export LLM=transformers BASE_MODEL=Qwen/Qwen3-4B-Instruct-2507 LORA_ADAPTER=adapters/indianlegal-qwen3-4b-lora
#   python -m indianlegal_llm.app.cli "Is privacy a fundamental right in India?"
from indianlegal_llm.config import Settings
from indianlegal_llm.pipeline import build_pipeline

pipe = build_pipeline(Settings(llm="transformers", base_model=SERVE_BASE, adapter=ADAPTER_DIR))
print("serving backend:", pipe.llm_backend)
answer = pipe.answer("Is privacy a fundamental right in India?")
print("refused:", answer.refused)
for c in answer.citations:
    print("-", c.reference)

---
**Do not commit** `data/`, `adapters/`, `outputs/`, or any `*.safetensors` — all gitignored.
Publish the adapter (MIT) to a model registry / HF Hub from the cloud host; the repo only
ever tracks code, configs, and this notebook (with outputs cleared).